# 🍎 Pixels-to-Macros — Food Segmentation Training

**Before running:**  
1. Runtime → Change runtime type → **GPU** (T4 is free, A100 is faster on Colab Pro)  
2. Connect to a runtime, then run cells top-to-bottom.

Outputs (checkpoints + metrics) are saved to **Google Drive** so they survive session resets.

## Cell 1 — Verify GPU

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Cell 2 — Mount Google Drive (saves checkpoints across sessions)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
OUTPUT_DIR = pathlib.Path('/content/drive/MyDrive/pixels-to-macros-training')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output dir:', OUTPUT_DIR)

## Cell 3 — Clone repo

In [ ]:
import pathlib, os

REPO_DIR = pathlib.Path('/content/pixels-to-macros')

if REPO_DIR.exists():
    print('Repo already cloned — pulling latest...')
    %cd /content/pixels-to-macros
    !git pull
else:
    !git clone https://github.com/JormayBusso/pixels-to-macros.git /content/pixels-to-macros
    %cd /content/pixels-to-macros

!ls training/

## Cell 4 — Install requirements

> This installs a CUDA-enabled PyTorch first, then the rest of `training/requirements.txt`.
> `coremltools` is excluded (CoreML export is macOS-only; do that step on your Mac later).

In [ ]:
import subprocess, sys

# Detect CUDA version from nvidia-smi
result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
cuda_tag = 'cu121'  # default for Colab T4 (CUDA 12.1)
if 'cuda_12.2' in result.stdout or 'cuda_12.2' in result.stderr:
    cuda_tag = 'cu122'
elif 'cuda_11' in result.stdout or 'cuda_11' in result.stderr:
    cuda_tag = 'cu118'
print(f'Installing PyTorch for {cuda_tag}...')

# Install CUDA-enabled PyTorch
!pip install -q --index-url https://download.pytorch.org/whl/{cuda_tag} \
    torch==2.2.0 torchvision==0.17.0

# Install remaining requirements, skipping coremltools (macOS-only)
!grep -v 'coremltools' /content/pixels-to-macros/training/requirements.txt > /tmp/colab_reqs.txt
!pip install -q -r /tmp/colab_reqs.txt

print('\nAll packages installed!')

## Cell 5 — (Optional) Install SegFormer support

Only needed if you plan to train with `--model segformer`. Skip otherwise.

In [ ]:
# Uncomment to install:
# !pip install -q transformers
print('Skipped (remove comment above to install SegFormer support)')

## Cell 6 — Download FoodSeg103 dataset (~1.3 GB)

Skipped automatically if the dataset already exists.

In [ ]:
%cd /content/pixels-to-macros

import pathlib
train_dir = pathlib.Path('data/FoodSeg103/Images/img_dir/train')

if train_dir.exists() and any(train_dir.glob('*.jpg')):
    print(f'Dataset already present ({len(list(train_dir.glob("*.jpg")))} train images). Skipping download.')
else:
    print('Downloading FoodSeg103...')
    !python scripts/download_foodseg103.py

## Cell 7 — Training config

Adjust these variables before starting training.

In [ ]:
# ── Training config ───────────────────────────────────────────────
MODEL         = 'mobilenet'    # 'mobilenet' | 'resnet101' | 'segformer'
EPOCHS        = 50
BATCH_SIZE    = 4              # reduce to 2 if OOM
IMG_SIZE      = 640
LR            = 2e-4
NUM_WORKERS   = 2
AMP           = True           # mixed precision (faster on GPU)
COMPILE       = False          # torch.compile — set True for PyTorch 2+ speedup
VAL_EVERY     = 1              # validate every N epochs
MULTIPLIER    = 10             # virtual training data multiplier
SAVE_BATCHES  = 200            # checkpoint every N batches
SAVE_SECS     = 300            # checkpoint every N seconds
DATA_DIR      = './data/FoodSeg103'
OUTPUT_DIR    = '/content/drive/MyDrive/pixels-to-macros-training'

print(f'Model: {MODEL} | Epochs: {EPOCHS} | Batch: {BATCH_SIZE} | AMP: {AMP}')
print(f'Output: {OUTPUT_DIR}')

## Cell 8 — Run training 🚀

Training auto-resumes from the last checkpoint if one exists in `OUTPUT_DIR`.  
Progress bars appear in real time below this cell.

In [ ]:
%cd /content/pixels-to-macros

amp_flag     = '--amp'     if AMP     else ''
compile_flag = '--compile' if COMPILE else ''

cmd = f"""
python training/train.py \\
  --data_dir {DATA_DIR} \\
  --output_dir {OUTPUT_DIR} \\
  --model {MODEL} \\
  --epochs {EPOCHS} \\
  --batch_size {BATCH_SIZE} \\
  --img_size {IMG_SIZE} \\
  --lr {LR} \\
  --num_workers {NUM_WORKERS} \\
  --val_every {VAL_EVERY} \\
  --virtual_train_multiplier {MULTIPLIER} \\
  --save_every_batches {SAVE_BATCHES} \\
  --save_every_secs {SAVE_SECS} \\
  {amp_flag} {compile_flag}
""".strip()

print('Running:\n', cmd, '\n')
!{cmd}

## Cell 9 — Live metrics watcher

Run this in a **separate cell** while Cell 8 is running to watch epoch-level results update every 10 seconds.  
Stop it with the ■ button or `Kernel → Interrupt`.

In [ ]:
import time, json, pathlib
from IPython.display import clear_output

metrics_path = pathlib.Path(OUTPUT_DIR) / 'metrics.json'
ckpt_path    = pathlib.Path(OUTPUT_DIR) / 'last_checkpoint.pth'
best_path    = pathlib.Path(OUTPUT_DIR) / 'best.pth'

print(f'Watching {metrics_path}  (updates every 10 s) — interrupt to stop\n')

while True:
    clear_output(wait=True)
    print(f'Watching {metrics_path}\n')

    if metrics_path.exists():
        try:
            data = json.loads(metrics_path.read_text())
            print(f'{'Epoch':>6}  {'Train Loss':>10}  {'Val Loss':>9}  {'mIoU':>7}  {'Pixel Acc':>10}')
            print('-' * 52)
            for row in data[-10:]:  # last 10 epochs
                print(
                    f"{row.get('epoch','-'):>6}  "
                    f"{row.get('train_loss', float('nan')):>10.4f}  "
                    f"{row.get('val_loss',   float('nan')):>9.4f}  "
                    f"{row.get('val_miou',   float('nan')):>7.4f}  "
                    f"{row.get('val_pixel_acc', float('nan')):>10.4f}"
                )
            best = max((r.get('val_miou', 0) for r in data), default=0)
            print(f'\nBest mIoU so far: {best:.4f}')
        except Exception as e:
            print(f'(metrics not ready yet: {e})')
    else:
        print('No metrics.json yet — training may still be in the first epoch.')

    ckpt_size = f'{ckpt_path.stat().st_size / 1e6:.1f} MB' if ckpt_path.exists() else 'not saved yet'
    best_size = f'{best_path.stat().st_size / 1e6:.1f} MB' if best_path.exists() else 'not saved yet'
    print(f'\nlast_checkpoint.pth : {ckpt_size}')
    print(f'best.pth            : {best_size}')

    time.sleep(10)

## Cell 10 — Verify saved files on Drive

In [ ]:
import pathlib, json

out = pathlib.Path(OUTPUT_DIR)
files = sorted(out.glob('*'))

print(f'Files in {out}:')
for f in files:
    size = f.stat().st_size / 1e6
    print(f'  {f.name:<35} {size:>8.2f} MB')

metrics_file = out / 'metrics.json'
if metrics_file.exists():
    data = json.loads(metrics_file.read_text())
    print(f'\nTotal epochs logged: {len(data)}')
    if data:
        best = max(data, key=lambda r: r.get('val_miou', 0))
        print(f'Best epoch: {best["epoch"]}  mIoU: {best.get("val_miou",0):.4f}  pixel_acc: {best.get("val_pixel_acc",0):.4f}')

## Tips

| Situation | Fix |
|---|---|
| OOM (out of memory) | Lower `BATCH_SIZE` to 2, or `IMG_SIZE` to 512 |
| Session reset | Re-run cells 2–3, then Cell 8 — training auto-resumes from the last checkpoint |
| Faster training | Set `MODEL = 'resnet101'` (stronger backbone); enable `COMPILE = True` on PyTorch 2+ |
| SegFormer | Run Cell 5, then set `MODEL = 'segformer'` |
| Export CoreML | After training, copy `best.pth` to your Mac and run `python training/export_coreml.py --checkpoint training/output/best.pth` |